# Clinicopathological and Molecular Characteristics of Second Primary Colorectal Cancer in Cancer Survivors including MSI-H Status and Anatomical Distribution Exploration with `mlcroissant`
This notebook provides a template for loading and exploring the FAIR^2 dataset using the `mlcroissant` library.

### Dataset Source
The dataset source is provided via a Croissant schema URL.

**Schema URL**:
`https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json`

In [ ]:
# Ensure `mlcroissant` library is installed
!pip install mlcroissant

## 1. Data Loading
Load metadata and records from the dataset using the `mlcroissant` library.

In [ ]:
import mlcroissant as mlc
import pandas as pd

# Define the Croissant schema URL
croissant_url = 'https://sen.science/doi/10.71728/senscience.qs2f-h81p/fair2.json'

# Load the dataset metadata
dataset = mlc.Dataset(croissant_url)
metadata = dataset.metadata

print(f"{metadata.name}: {metadata.description}")

## 2. Data Overview
Review available record sets and their fields using their `@id` identifiers.

In [ ]:
# List all record sets by @id and their fields by @id
record_sets = list(dataset.record_sets.values())
print("Record sets in the dataset:")
for rs in record_sets:
    print(f"- RecordSet @id: {rs.id}")
    fields = rs.fields
    print("  Fields:")
    for field in fields:
        print(f"    - Field @id: {field.id} (name: {field.name}, type: {field.data_type})")

## 3. Data Extraction
Load data from a specific record set into a DataFrame for analysis using its `@id`. All entities are referenced by their `@id` fields.

In [ ]:
# Collect all record set @ids
record_set_ids = [rs.id for rs in record_sets]
print("Available record set @ids:")
for rid in record_set_ids:
    print(" -", rid)

dataframes = {}

for record_set_id in record_set_ids:
    records = list(dataset.records(record_set=record_set_id))
    if records:
        dataframes[record_set_id] = pd.DataFrame(records)

# Pick the first record set with records for demonstration
if dataframes:
    selected_record_set_id = list(dataframes.keys())[0]
    print(f"\nColumns in record set {selected_record_set_id}:")
    print(dataframes[selected_record_set_id].columns.tolist())
    dataframes[selected_record_set_id].head()
else:
    print("No record sets with data were found.")

## 4. Exploratory Data Analysis (EDA)
Apply basic EDA: filtering, normalizing, and grouping using field `@id`s. We'll select a numeric field (if available), filter by a threshold, normalize the field, and group by a categorical attribute.

In [ ]:
# Check for numeric fields in the selected DataFrame
import numpy as np

df = dataframes[selected_record_set_id]
numeric_field_id = None
group_field_id = None

# Attempt to find a numeric field using @id
for field in dataset.record_sets[selected_record_set_id].fields:
    col_id = field.id
    if pd.api.types.is_numeric_dtype(df.get(col_id, pd.Series([],dtype=float))):
        numeric_field_id = col_id
        break

# Attempt to find a categorical/group field (string/object) using @id
for field in dataset.record_sets[selected_record_set_id].fields:
    col_id = field.id
    if pd.api.types.is_string_dtype(df.get(col_id, pd.Series([],dtype=str))):
        group_field_id = col_id
        break

if numeric_field_id is not None:
    print(f"Using numeric field: {numeric_field_id}")
    if numeric_field_id in df.columns:
        try:
            df[numeric_field_id] = pd.to_numeric(df[numeric_field_id], errors='coerce')
        except Exception:
            pass
        threshold = np.nanquantile(df[numeric_field_id], 0.5)  # median value as threshold
        filtered_df = df[df[numeric_field_id] > threshold]
        print(f"Filtered records with {numeric_field_id} > {threshold}:")
        print(filtered_df.head())

        # Normalization
        filtered_df[f"{numeric_field_id}_normalized"] = (
            filtered_df[numeric_field_id] - filtered_df[numeric_field_id].mean()
        ) / filtered_df[numeric_field_id].std()
        print(f"\nNormalized {numeric_field_id} for filtered records:")
        print(filtered_df[[numeric_field_id, f"{numeric_field_id}_normalized"]].head())

        # Grouping
        if group_field_id is not None and group_field_id in filtered_df.columns:
            grouped_df = filtered_df.groupby(group_field_id)[numeric_field_id].mean().reset_index()
            print(f"\nGrouped mean of {numeric_field_id} by {group_field_id}:")
            print(grouped_df.head())
        else:
            print("No suitable group field found for grouping.")
    else:
        print(f"Numeric field '{numeric_field_id}' not present in DataFrame columns.")
else:
    print("No numeric field found in the dataset for EDA.")

## 5. Visualization
Visualize the distribution of the selected numeric field and relationship with the group field (if available).

In [ ]:
import matplotlib.pyplot as plt
import seaborn as sns

if numeric_field_id is not None and numeric_field_id in df.columns:
    plt.figure(figsize=(8,4))
    sns.histplot(df[numeric_field_id].dropna(), kde=True)
    plt.title(f"Distribution of {numeric_field_id}")
    plt.xlabel(numeric_field_id)
    plt.show()
    
    if group_field_id is not None and group_field_id in df.columns:
        plt.figure(figsize=(10,4))
        sns.boxplot(x=df[group_field_id], y=df[numeric_field_id])
        plt.title(f"{numeric_field_id} by {group_field_id}")
        plt.xlabel(group_field_id)
        plt.ylabel(numeric_field_id)
        plt.xticks(rotation=45)
        plt.show()
else:
    print("No numeric field found for visualization.")

## 6. Conclusion
In this notebook, we demonstrated how to load, explore, and process data from the FAIR^2 dataset using `mlcroissant`.

- All exploration was performed using stable `@id` references for record sets and fields.
- We loaded the available record sets, listed fields, and examined records for analysis.
- Exploratory data analysis included basic filtering, normalization, and grouping by categorical attributes.
- Data visualization enabled fast insights into numeric data distributions and groupings.

For in-depth studies, combine these steps with domain knowledge and advanced statistical or machine learning pipelines.